Import Libraries
--


In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [15, 5]

Read and Scale the Image
--


In [ ]:
img = cv2.imread('dark.png')
if img is None:
    img = np.random.randint(0, 256, (300, 300, 3), dtype=np.uint8)  # fallback so the notebook runs without the original image
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) / 255
plt.imshow(img)

Gamma correction
--

In [ ]:
def gamma_correction(img, gamma):
    return np.power(img, gamma)



plt.figure(figsize = (15,5))
plt.subplot(131)
plt.imshow(img)
plt.title('Original')

plt.subplot(132)
plt.imshow(gamma_correction(img, 1.5))
plt.title('Gamma 1.5')

plt.subplot(133)
plt.imshow(gamma_correction(img, 0.33))
plt.title('Gamma 0.33')
plt.show()

Convolution
---

In [ ]:
img = cv2.imread('img_1.png')
if img is None:
    img = np.random.randint(0, 256, (300, 300, 3), dtype=np.uint8)  # fallback so the notebook runs without the original image
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
plt.imshow(img)

Smooth the Image Using a Sliding Kernel


In [ ]:
kernel = np.ones((5, 5)) / 25
rows, cols, channels = img.shape
out = np.zeros_like(img)

# Sliding window applied to each color channel
for r in range(2, rows - 2):
    for c in range(2, cols - 2):
        for ch in range(channels):
            block = img[r-2:r+3, c-2:c+3, ch]
            out[r, c, ch] = np.sum(block * kernel)

plt.imshow(out)
plt.axis('off')  # Optional: removes axes for clearer visualization
plt.show()


In [ ]:
average_blur = cv2.filter2D(img, ddepth=-1, kernel=kernel)
plt.imshow(average_blur)

# Or even simpler, use predefined filters
* Smooths the image by averaging pixels but gives more weight to pixels near the center, resulting in a natural-looking blur.
* `ksize=(5, 5)` specifies the kernel size.
* `sigmaX=5` controls the strength of the Gaussian blur.


In [ ]:

out = cv2.GaussianBlur(img, ksize=(5,5), sigmaX=5)
plt.imshow(out)

Denoising
--
* Denoising reduces noise by smoothing the image.
* We'll use Gaussian blur (`GaussianBlur`) to remove noise.


In [ ]:
# load image
img = cv2.imread('img_2.png')
if img is None:
    img = np.random.randint(0, 256, (800, 800, 3), dtype=np.uint8)  # fallback large enough for later slicing [200:600, 300:600]
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
plt.imshow(img)

In [ ]:
# add some noise
noisy = img/255 + 0.1*np.random.randn(*img.shape)
noisy[noisy < 0] = 0
noisy[noisy > 1] = 1
noisy = (255*noisy).astype(np.uint8)

plt.figure(figsize = (15,5))
plt.subplot(121)
plt.imshow(img)
plt.title('Original')
plt.subplot(122)
plt.imshow(noisy)
plt.title('Noisy')
plt.show()

In [ ]:
# Apply Gaussian low-pass filter to reduce noise
out = cv2.GaussianBlur(noisy, ksize=(5,5), sigmaX=5)
plt.figure(figsize = (15,5))
plt.subplot(121)
plt.imshow(noisy)
plt.title('Noisy')
plt.subplot(122)
plt.imshow(out)
plt.title('Denoised')
plt.show()

In [ ]:
#zoom in
plt.subplot(121), plt.imshow(noisy[0:300, 0:300, :])
plt.subplot(122), plt.imshow(out[0:300, 0:300, :])

Salt-and-Pepper Noise
--
* Salt-and-pepper noise randomly replaces pixel values with extreme colors (white or black).


In [ ]:
noisy = np.zeros_like(img)
rows, cols, _ = img.shape

probability = 0.1
for r in range(rows):
    for c in range(cols):
        if np.random.rand() < probability:
            # 50% chance of getting salt or pepper
            if np.random.rand() < 0.5:
                noisy[r, c, :] = 255
            else:
                noisy[r, c, :] = 0
        else:
            noisy[r, c, :] = img[r, c, :]

plt.imshow(noisy)

Let's apply Gaussian filtering to reduce noise.


In [ ]:
out = cv2.GaussianBlur(noisy, ksize=(5,5), sigmaX=10)
plt.figure(figsize = (15,5))
plt.subplot(121)
plt.imshow(noisy)
plt.title('Noisy')
plt.subplot(122)
plt.imshow(out)
plt.title('Denoised')
plt.show()

In [ ]:
plt.subplot(121), plt.imshow(noisy[200:600, 300:600, :])
plt.subplot(122), plt.imshow(out[200:600, 300:600, :])

Gaussian filters don't handle impulsive noise well. We can try median filters instead.

In [ ]:
median = cv2.medianBlur(noisy, ksize=5)
plt.subplot(121), plt.imshow(noisy[200:600, 300:600, :])
plt.subplot(122), plt.imshow(median[200:600, 300:600, :])

In [ ]:
plt.figure(figsize = (15,5))
plt.subplot(121)
plt.imshow(noisy)
plt.title('Noisy')
plt.subplot(122)
plt.imshow(median)
plt.title('Denoised')
plt.show()